<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part F: Appendices</h2>
<h2>Notebook F01c: Preparing the External Datasets</h2>
</div>

Besides the datasets we prepared for the course and host on Hugging Face, a few notebooks use
**external datasets** that come straight from their original publisher. This notebook gets them onto
your machine and checks that they are usable.

Run it once before starting the notebooks that need them. You do not have to run it again unless you
want to refresh the data.

---

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Dataset Overview</h3>
</div>

| Dataset | Source | Licence | Used in | How to get it |
|---------|--------|---------|---------|---------------|
| Individual Household Electric Power Consumption | [UCI](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) | CC BY 4.0 | A03 (missing data) | Automatic, in this notebook |
| Air Quality | [UCI](https://archive.ics.uci.edu/dataset/360/air+quality) | CC BY 4.0 | A04 (outliers) | Automatic, in this notebook |
| Rossmann Store Sales | [Kaggle](https://www.kaggle.com/competitions/rossmann-store-sales) | Competition rules | A04 (outliers) | Manual, see section 5 |

Everything lands under `data/external/`, in one folder per dataset. That folder is ignored by git, so
the files stay on your machine and never end up in a commit.

> **Why is one of them manual?** The two UCI datasets are published under a licence that lets anyone
> redistribute them, so we can fetch them for you. The Rossmann data is competition data: Kaggle asks
> every user to accept the competition rules personally before downloading, which is not something a
> script can do on your behalf.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. A Small Download Helper</h3>
</div>

Both UCI datasets are published as ZIP archives. The helper below downloads an archive into memory,
extracts only the files we actually need, and writes them to the right folder.

It checks first whether the files are already there. Re-running this notebook therefore costs
nothing, and it will not overwrite data you already have.

In [ ]:
import zipfile
from io import BytesIO
from urllib.error import URLError
from urllib.request import urlopen

import nb_config


def human_size(path):
    """Format a file size the way a person would write it."""
    size = path.stat().st_size
    return f"{size / 1e6:.1f} MB" if size >= 1e6 else f"{size / 1e3:.0f} KB"


def prepare_from_zip(url, target_dir, members, label):
    """Download a ZIP archive and extract `members` into `target_dir`.

    Files that are already present are left untouched.
    """
    target_dir.mkdir(parents=True, exist_ok=True)

    missing = [m for m in members if not (target_dir / m).exists()]
    if not missing:
        print(f"✅  {label}: already present in {target_dir}")
        return

    print(f"Downloading {label} ...")
    print(f"  from {url}")
    try:
        with urlopen(url, timeout=120) as response:
            payload = BytesIO(response.read())
    except URLError as error:
        print(f"  ❌  Download failed: {error}")
        print("      Check your internet connection, or download the file manually")
        print(f"      from the source link in section 1 and unzip it into {target_dir}")
        return

    with zipfile.ZipFile(payload) as archive:
        for member in missing:
            archive.extract(member, target_dir)
            print(f"  extracted {member}  ({human_size(target_dir / member)})")

    print(f"✅  {label}: ready")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Household Power Consumption</h3>
</div>

Minute-level electric power measurements from a single household near Paris, recorded between
December 2006 and November 2010. It contains genuine gaps where the measuring equipment was down,
which is exactly what makes it a good dataset for practising how to deal with missing values.

The archive is about 20 MB; unpacked the file is around 127 MB, so the download takes a moment.

In [ ]:
prepare_from_zip(
    url="https://archive.ics.uci.edu/static/public/235/"
        "individual+household+electric+power+consumption.zip",
    target_dir=nb_config.HOUSEHOLD_POWER_DIR,
    members=["household_power_consumption.txt"],
    label="Household Power Consumption",
)

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Air Quality</h3>
</div>

Hourly readings from a chemical multisensor device deployed in an Italian city between March 2004 and
February 2005. The sensors produce occasional implausible values, and missing readings are tagged
with `-200` rather than being left empty, which makes this a realistic dataset for outlier work.

The archive also contains an Excel version of the same data. We only extract the CSV.

In [ ]:
prepare_from_zip(
    url="https://archive.ics.uci.edu/static/public/360/air+quality.zip",
    target_dir=nb_config.AIR_QUALITY_DIR,
    members=["AirQualityUCI.csv"],
    label="Air Quality",
)

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Rossmann Store Sales</h3>
</div>

Daily sales for 1,115 Rossmann drug stores, published for a Kaggle competition. Kaggle requires you
to accept the competition rules with your own account before the data can be downloaded, so this one
needs a few manual steps.

**What to do**

1. Create a Kaggle account, if you do not have one, and sign in.
2. Open the [Rossmann Store Sales competition](https://www.kaggle.com/competitions/rossmann-store-sales)
   and accept the rules on the *Rules* tab.
3. Go to the *Data* tab and download `rossmann-store-sales.zip`.
4. Unzip it into this folder of the repository:

   ```
   data/external/Rossmann_Store_Sales_Dataset/
   ```

   You should end up with `train.csv`, `test.csv`, `store.csv`, and `sample_submission.csv` directly
   inside it, not in a further subfolder.

Run the cell below afterwards to confirm the files ended up in the right place.

In [ ]:
expected = {
    "train.csv": nb_config.ROSSMANN_TRAIN_PATH,
    "store.csv": nb_config.ROSSMANN_STORE_PATH,
}

found = {name: path.exists() for name, path in expected.items()}

for name, path in expected.items():
    if found[name]:
        print(f"✅  {name}  ({human_size(path)})")
    else:
        print(f"❌  {name} not found at {path}")

if all(found.values()):
    print("\nRossmann Store Sales: ready")
else:
    print(f"\nFollow the steps above, then unzip into:\n  {nb_config.ROSSMANN_DIR}")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. Validation</h3>
</div>

A last check that every file is present and actually readable. The two UCI files use European
conventions that trip up the default `read_csv` settings, so we pass the separator and the decimal
mark explicitly, exactly as the course notebooks do.

In [ ]:
import pandas as pd


def validate(label, path, **read_kwargs):
    print(f"--- {label} ---")

    if not path.exists():
        print(f"  ❌  Not found at {path}")
        print("      Re-run the section above that prepares this dataset.\n")
        return False

    print(f"  ✅  File exists  ({human_size(path)})")

    try:
        head = pd.read_csv(path, nrows=5, **read_kwargs)
    except Exception as error:
        print(f"  ❌  Could not be read: {error}\n")
        return False

    print(f"  ✅  Readable  |  {len(head.columns)} columns")
    print(f"      {list(head.columns)[:6]}")
    print()
    return True


results = [
    validate(
        "Household Power Consumption",
        nb_config.HOUSEHOLD_POWER_PATH,
        sep=";",
        decimal=".",
        low_memory=False,
    ),
    validate(
        "Air Quality",
        nb_config.AIR_QUALITY_PATH,
        sep=";",
        decimal=",",
    ),
    validate(
        "Rossmann Store Sales",
        nb_config.ROSSMANN_TRAIN_PATH,
        low_memory=False,
    ),
]

ready = sum(results)
print(f"{ready} of {len(results)} external datasets ready.")
if ready < len(results):
    print("The notebooks that use the missing ones will not run until you prepare them.")

---

The external datasets are ready. Notebook
[A03](./A03_Handling_missing_data.ipynb) uses the household power consumption data, and
[A04](./A04_Handling_outliers.ipynb) uses the air quality and Rossmann data.

If you are looking for the datasets we host ourselves, they are prepared in
[F01a](./F01a_Preparing_OPS_datasets.ipynb) and [F01b](./F01b_Preparing_CDC_dataset.ipynb).
An overview of every dataset in the course is kept in [`datasets.md`](../data/datasets.md).